# Kaggle — fine-tune YOLO11l qua đêm

1. Upload dataset lên Hugging Face bằng scripts/upload_finetune_dataset.py trước.
2. Import notebook này vào Kaggle, bật **Internet** và **GPU**.
3. Trong Secrets, thêm **HF_TOKEN** có quyền đọc repo và cho phép notebook truy cập.
4. Kiểm tra cấu hình cell bên dưới. Chạy thử đến cell xem nhãn trước nếu cần.
5. Chọn **Save Version → Save & Run All** và chọn GPU cho phiên chạy nếu giao diện hỏi.
   Chờ phiên background chuyển sang Running rồi có thể đóng trình duyệt.
   Save & Run All chạy lại từ đầu trong phiên riêng, không tiếp tục cell interactive đang chạy.
6. Khi hoàn thành, tải weights/ZIP trong Output của version đó.

Kết quả lưu ở /kaggle/working/yolo-training, ảnh/cache ở /tmp để không đóng gói ảnh
vào notebook Output. Chỉ dùng GPU 0, kể cả máy có hai GPU; không tự cộng VRAM.
Giới hạn thời gian training mặc định 8 giờ để chừa thời gian cho download/validation;
đây là cấu hình người dùng, không phải cam kết quota Kaggle. Kiểm tra quota tài khoản trước chạy.


In [ ]:
%pip install "ultralytics>=8.3,<8.4" "huggingface_hub>=0.23" "Pillow>=10,<13" pyyaml


In [ ]:
import os, sys, json, zipfile, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import torch, yaml
from huggingface_hub import HfApi, hf_hub_download
from ultralytics import YOLO

assert sys.version_info >= (3, 11), 'Notebook cần Python >=3.11'
assert torch.cuda.is_available(), 'Bật GPU trong Settings trước khi Save & Run All'
print('GPU:', torch.cuda.get_device_name(0))
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
REPO_ID = 'ericng17/hcm-traffic-yolo-v20'
DATA_ROOT = Path('/tmp/hcm-yolo-v20')
PROJECT = Path('/kaggle/working/yolo-training')
RUN_NAME = 'yolo11l-v20-run1'
EPOCHS = 100
BATCH = 4                   # Nếu OOM, thử 2 cho run mới
MAX_TRAIN_HOURS = 8.0        # Giảm nếu quota còn ít; không đảm bảo đủ 100 epoch
RESUME = False
# Nếu resume từ Output version cũ đã Add Input, điền thư mục chứa run và source.json:
RESUME_FROM = None          # Ví dụ '/kaggle/input/previous-output/yolo-training'
if RESUME and RESUME_FROM:
    previous = Path(RESUME_FROM)
    assert (previous / RUN_NAME / 'weights/last.pt').is_file()
    PROJECT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(previous / RUN_NAME, PROJECT / RUN_NAME, dirs_exist_ok=True)
    shutil.copy2(previous / f'{RUN_NAME}-source.json', PROJECT / f'{RUN_NAME}-source.json')


In [ ]:
PROJECT.mkdir(parents=True, exist_ok=True)
provenance = PROJECT / f'{RUN_NAME}-source.json'
api = HfApi(token=os.environ['HF_TOKEN'])
if provenance.exists():
    source = json.loads(provenance.read_text())
    assert source['repo_id'] == REPO_ID
    revision = source['revision']
else:
    revision = api.repo_info(REPO_ID, repo_type='dataset').sha
    provenance.write_text(json.dumps({'repo_id': REPO_ID, 'revision': revision}, indent=2))
archive = hf_hub_download(REPO_ID, 'dataset.zip', repo_type='dataset', revision=revision, cache_dir='/tmp/hf-cache')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        target = (DATA_ROOT / info.filename).resolve()
        assert target.is_relative_to(DATA_ROOT.resolve()), 'Unsafe ZIP path'
    z.extractall(DATA_ROOT)
cfg = yaml.safe_load((DATA_ROOT / 'data.yaml').read_text())
cfg.update(path=str(DATA_ROOT), train='train/images', val='valid/images', test='test/images')
(DATA_ROOT / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print(yaml.safe_dump(cfg, sort_keys=False))


## Kiểm tra lại định dạng và ảnh trước train

In [ ]:
"""Audit YOLO detection labels, image integrity and exact split overlap."""
import argparse
import hashlib
import json
import math
from collections import Counter, defaultdict
from pathlib import Path

import yaml
from PIL import Image


def audit(root):
    root = Path(root).resolve()
    cfg = yaml.safe_load((root / 'data.yaml').read_text())
    names = cfg['names']
    if isinstance(names, dict):
        names = [names[i] for i in range(len(names))]
    errors, warnings, splits = [], [], {}
    if cfg.get('nc', len(names)) != len(names):
        errors.append('nc differs from names length')
    hashes = defaultdict(list)
    for split, folder in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
        image_dir, label_dir = root / folder / 'images', root / folder / 'labels'
        if not image_dir.is_dir() or not label_dir.is_dir():
            errors.append(f'{split}: missing images/labels directory')
            continue
        configured = (root / cfg[split]).resolve()
        if configured != image_dir:
            errors.append(f'{split}: YAML resolves to {configured}, expected {image_dir}')
        images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        counts, sizes = Counter(), Counter()
        empty, missing, boxes, duplicate_rows = 0, 0, 0, 0
        stems = Counter(p.stem for p in images)
        for stem, count in stems.items():
            if count > 1:
                errors.append(f'{split}: multiple images share label stem {stem}')
        for image in images:
            relative = image.relative_to(root).as_posix()
            try:
                with Image.open(image) as im:
                    im.load()
                    width, height = im.size
                    sizes[f'{width}x{height}'] += 1
            except Exception as exc:
                errors.append(f'{relative}: unreadable image: {exc}')
                continue
            hashes[hashlib.sha256(image.read_bytes()).hexdigest()].append(relative)
            label = label_dir / (image.stem + '.txt')
            if not label.exists():
                missing += 1
                errors.append(f'{relative}: missing label (review before treating as background)')
                continue
            lines = [s for s in label.read_text().splitlines() if s.strip()]
            empty += not lines
            seen = set()
            for line_no, line in enumerate(lines, 1):
                location = f'{label.relative_to(root)}:{line_no}'
                try:
                    fields = line.split()
                    if len(fields) != 5:
                        raise ValueError('expected class xc yc w h (5 fields)')
                    cls, x, y, w, h = map(float, fields)
                    if not all(math.isfinite(v) for v in [cls, x, y, w, h]):
                        raise ValueError('non-finite value')
                    if not cls.is_integer() or not 0 <= cls < len(names):
                        raise ValueError('invalid class ID')
                    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                        raise ValueError('invalid normalized xywh')
                    if x-w/2 < -1e-5 or y-h/2 < -1e-5 or x+w/2 > 1+1e-5 or y+h/2 > 1+1e-5:
                        raise ValueError('box extends outside image')
                    row = (cls, x, y, w, h)
                    duplicate_rows += row in seen
                    seen.add(row)
                    counts[names[int(cls)]] += 1
                    boxes += 1
                except ValueError as exc:
                    errors.append(f'{location}: {exc}')
        orphan = [str(p.relative_to(root)) for p in label_dir.glob('*.txt') if p.stem not in stems]
        errors.extend(f'orphan label: {p}' for p in orphan)
        if duplicate_rows:
            warnings.append(f'{split}: {duplicate_rows} duplicate annotation rows')
        if empty:
            warnings.append(f'{split}: {empty} empty labels; manually verify background images')
        splits[split] = dict(images=len(images), boxes=boxes, class_counts=dict(counts),
                             empty_labels=empty, missing_labels=missing, image_sizes=dict(sizes),
                             duplicate_label_rows=duplicate_rows)
    duplicates = [paths for paths in hashes.values() if len(paths) > 1]
    cross_split = [paths for paths in duplicates if len({p.split('/')[0] for p in paths}) > 1]
    if cross_split:
        warnings.append(f'{len(cross_split)} exact-image hash groups cross dataset splits')
    return dict(names=names, splits=splits, errors=errors, warnings=warnings,
                exact_duplicate_groups=duplicates, cross_split_duplicates=cross_split,
                scope='Format/integrity only; no semantic label or near-duplicate verification')



report = audit(DATA_ROOT)
print(json.dumps(report['splits'], indent=2))
print('Warnings:', report['warnings'])
assert not report['errors'], report['errors'][:20]

(PROJECT / f'{RUN_NAME}-audit.json').write_text(json.dumps(report, indent=2, ensure_ascii=False))


In [ ]:
# Xem annotation gốc (không phải dự đoán) trên ảnh từng split.
from PIL import Image, ImageDraw
from IPython.display import display
for split in ['train', 'valid', 'test']:
    candidates = sorted((DATA_ROOT / split / 'images').glob('*'))
    for p in candidates[:2]:
        image = Image.open(p).convert('RGB')
        draw = ImageDraw.Draw(image)
        w, h = image.size
        label = DATA_ROOT / split / 'labels' / (p.stem + '.txt')
        for line in label.read_text().splitlines():
            if not line.strip(): continue
            cls, x, y, bw, bh = map(float, line.split())
            box = ((x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h)
            draw.rectangle(box, outline='lime', width=2)
            draw.text(box[:2], cfg['names'][int(cls)], fill='yellow')
        print(split, p.name)
        display(image)


In [ ]:
import ultralytics
(PROJECT / f'{RUN_NAME}-environment.json').write_text(json.dumps({
    'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0), 'dataset_revision': revision,
}, indent=2))
last = PROJECT / RUN_NAME / 'weights/last.pt'
if RESUME:
    assert last.exists(), 'Không có last.pt để resume'
    model = YOLO(str(last))
    model.train(resume=True, time=MAX_TRAIN_HOURS, device=0)
else:
    assert not (PROJECT / RUN_NAME).exists(), 'Đổi RUN_NAME hoặc chọn RESUME=True'
    model = YOLO('yolo11l.pt')
    model.train(data=str(DATA_ROOT / 'data.yaml'), imgsz=1024,
                epochs=EPOCHS, time=MAX_TRAIN_HOURS, patience=20, batch=BATCH, device=0,
                workers=2, cache=False, seed=42, deterministic=True,
                mixup=0.0, close_mosaic=10, save=True, save_period=10,
                project=str(PROJECT), name=RUN_NAME)

print('Training output:', PROJECT / RUN_NAME)


## Validation và test cuối

Dùng validation để chọn cấu hình. Chỉ bật test sau khi chốt phương pháp.
Nếu gặp lỗi/runtime bị kill cứng, cell cuối có thể không chạy; weights đã ghi dưới
/kaggle/working vẫn là checkpoint quan trọng. Chỉ resume được từ epoch đã hoàn thành.
Kaggle Output không phải ổ đĩa bền vững cho mọi tình huống ngắt; hãy kiểm tra Output của
version đã lưu. Notebook không tự upload weights lên Hugging Face.


In [ ]:
best = YOLO(str(PROJECT / RUN_NAME / 'weights/best.pt'))
metrics = best.val(data=str(DATA_ROOT / 'data.yaml'), split='val', imgsz=1024,
                   batch=BATCH, device=0, max_det=1000,
                   project=str(PROJECT), name=RUN_NAME+'-val')
print(metrics.results_dict)
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    test_metrics = best.val(data=str(DATA_ROOT / 'data.yaml'), split='test',
                            imgsz=1024, batch=BATCH, device=0, max_det=1000,
                            project=str(PROJECT), name=RUN_NAME+'-test')
    print(test_metrics.results_dict)
print('Weights trong Kaggle Output:', PROJECT / RUN_NAME / 'weights')


In [ ]:
# ZIP gọn gồm weights, log, biểu đồ, metadata; không chứa ảnh dataset.
archive = shutil.make_archive('/kaggle/working/yolo-training-results', 'zip', PROJECT)
from IPython.display import FileLink, display
display(FileLink(archive))
print('Best:', PROJECT / RUN_NAME / 'weights/best.pt')
print('Resume:', PROJECT / RUN_NAME / 'weights/last.pt')


Tài liệu: https://docs.ultralytics.com/modes/train/ và https://docs.ultralytics.com/datasets/detect/
